In [2]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import evaluate

warnings.filterwarnings("ignore")

c:\Users\fongx\Downloads\MSDS\CSC753M (Machine Learning)\final-project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
data_path = "./output/01_output_clean_data.jsonl"

In [4]:
df = pd.read_json(data_path, lines=True)

In [5]:
print(df.shape)
df.head()

(201573, 4)


,row_id,headline,category,text_combined
0,1,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,23 Of The Funniest Tweets About Cats And Dogs ...
1,2,The Funniest Tweets From Parents This Week (Se...,PARENTING,The Funniest Tweets From Parents This Week (Se...
2,3,Puerto Ricans Desperate For Water After Hurric...,WORLD NEWS,Puerto Ricans Desperate For Water After Hurric...
3,4,How A New Documentary Captures The Complexity ...,ARTS & CULTURE,How A New Documentary Captures The Complexity ...
4,5,Biden At UN To Call Russian War An Affront To ...,WORLD NEWS,Biden At UN To Call Russian War An Affront To ...


## TRAIN-VAL-TEST SPLIT

In [6]:
SEED = 2026

train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df["category"], random_state=SEED
)
train_df, val_df = train_test_split(
    train_df, test_size=0.20, stratify=train_df["category"], random_state=SEED
)

In [7]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(129006, 4)
(32252, 4)
(40315, 4)


In [8]:
for df in [train_df, val_df, test_df]:
    print(f"Count of unique categs: {df.category.nunique()}")
    print(df["category"].value_counts(normalize=True))
    print("\n")

Count of unique categs: 25
category
POLITICS          0.176527
WELLNESS          0.122173
ENTERTAINMENT     0.086112
PARENTING         0.062803
STYLE & BEAUTY    0.059168
TRAVEL            0.049091
WORLD NEWS        0.047323
FOOD & DRINK      0.041843
BUSINESS          0.038409
QUEER VOICES      0.031464
COMEDY            0.026735
SPORTS            0.025193
BLACK VOICES      0.022712
HOME & LIVING     0.021270
SCIENCE & TECH    0.021139
ENVIRONMENT       0.020139
ARTS & CULTURE    0.019449
WEDDINGS          0.018123
CRIME             0.017658
WOMEN             0.017294
IMPACT            0.017278
DIVORCE           0.016984
MEDIA             0.014588
WEIRD NEWS        0.013782
RELIGION          0.012744
Name: proportion, dtype: float64


Count of unique categs: 25
category
POLITICS          0.176547
WELLNESS          0.122194
ENTERTAINMENT     0.086103
PARENTING         0.062818
STYLE & BEAUTY    0.059159
TRAVEL            0.049113
WORLD NEWS        0.047315
FOOD & DRINK      0.041827
BU

In [9]:
train_path = "./output/02_train.jsonl"
train_df.to_json(train_path, orient="records", lines=True)

In [10]:
val_path = "./output/02_val.jsonl"
val_df.to_json(val_path, orient="records", lines=True)

In [11]:
test_path = "./output/02_test.jsonl"
test_df.to_json(test_path, orient="records", lines=True)

## Create the y_encoder

In [12]:
from sklearn.preprocessing import LabelEncoder

y_encoder = LabelEncoder()
y_train = y_encoder.fit_transform(train_df["category"])
y_test = y_encoder.transform(test_df["category"])
print(y_train[:10])
print(y_encoder.classes_)
print(len(y_encoder.classes_))

[ 7 17  7 22 22 22 13  1  4  6]
['ARTS & CULTURE' 'BLACK VOICES' 'BUSINESS' 'COMEDY' 'CRIME' 'DIVORCE'
 'ENTERTAINMENT' 'ENVIRONMENT' 'FOOD & DRINK' 'HOME & LIVING' 'IMPACT'
 'MEDIA' 'PARENTING' 'POLITICS' 'QUEER VOICES' 'RELIGION' 'SCIENCE & TECH'
 'SPORTS' 'STYLE & BEAUTY' 'TRAVEL' 'WEDDINGS' 'WEIRD NEWS' 'WELLNESS'
 'WOMEN' 'WORLD NEWS']
25


In [13]:
# Save pickled y encoder
import pickle

with open(
    "./artifacts/02_y_encoder.pkl",
    "wb",
) as f:
    pickle.dump(y_encoder, f)